# Known-k independent: техническая поправка после открытия
Это не новый закрытый тест: набор уже открыт, seed 7026 сохранён. Поправка устраняет чрезмерную проверку численно нестабильного вспомогательного S1b. Исходные freeze/seal/open/seed_7026 не переписываются; новое обучение, генерация и подбор запрещены. Выполняй кодовые ячейки вручную, пропуская защитные STOP-ячейки.

In [ ]:
from pathlib import Path
import importlib.util, re, subprocess, sys
if importlib.util.find_spec('google.colab') is None:
    raise RuntimeError('Этот notebook запускается в Google Colab')
if any(name == 'diffusion_sources' or name.startswith('diffusion_sources.') or name.startswith('scripts.known_k_independent') for name in sys.modules):
    raise RuntimeError('Перезапусти среду Colab перед setup: старый Python-код уже импортирован')
from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/diffusion-sources-known-k-independent')
REMOTE = 'https://github.com/1habibi/diffusion-sources-localization-.git'
BRANCH = 'experiment/known-k-independent-amendment'
ORIGINAL_REVISION = '1b3b90f8deac2b936a05593a14c0034edf242f6a'
PINNED_REVISION = input('Вставь проверенный SHA-40 поправки из инструкции: ').strip().lower()
if not re.fullmatch(r'[0-9a-f]{40}', PINNED_REVISION):
    raise ValueError('Ожидается полный 40-значный Git SHA')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REMOTE, str(PROJECT)], check=True)
else:
    if not (PROJECT / '.git').exists():
        raise RuntimeError(f'{PROJECT} существует, но не является Git-клоном')
    clean = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True).strip()
    current = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
    if clean or current not in (ORIGINAL_REVISION, PINNED_REVISION):
        raise RuntimeError('Клон изменён или имеет неизвестную ревизию; ничего не перезаписываем')
    subprocess.run(['git', '-C', str(PROJECT), 'fetch', '--no-tags', 'origin', BRANCH], check=True)
    fetched = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
    if fetched != PINNED_REVISION:
        raise RuntimeError(f'Ветка указывает на {fetched}, а ожидалось {PINNED_REVISION}; остановись')
    if current != PINNED_REVISION:
        subprocess.run(['git', '-C', str(PROJECT), 'checkout', '--detach', PINNED_REVISION], check=True)
REVISION = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
if REVISION != PINNED_REVISION:
    raise RuntimeError(f'Git revision {REVISION} не совпадает с проверенным {PINNED_REVISION}')
packages = {'numpy': 'numpy', 'networkx': 'networkx', 'scipy': 'scipy', 'sklearn': 'scikit-learn', 'yaml': 'PyYAML', 'tqdm': 'tqdm', 'torch_geometric': 'torch-geometric'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT), '--no-deps'], check=True)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT / 'src'))
import torch
from scripts import known_k_independent_amendment as AMENDMENT
from scripts.known_k_independent_artifacts import IndependentKnownKPaths
from scripts.known_k_independent_amendment import freeze_amendment, run_amended_seed, save_amended_summary
if not Path(AMENDMENT.__file__).resolve().is_relative_to(PROJECT.resolve()):
    raise RuntimeError('Kernel импортировал другую копию проекта; перезапусти среду')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Project:', PROJECT, 'revision:', REVISION, 'device:', DEVICE)
print('Setup загрузил только код поправки. Старые артефакты и метрики не изменены.')


In [ ]:
ROOT = globals().get('ROOT_OVERRIDE', Path('/content/drive/MyDrive/diffusion-sources'))
AMENDMENT_NOTEBOOK = PROJECT / 'notebooks/colab_known_k_independent_amendment.ipynb'
PATHS = IndependentKnownKPaths(
    repo=PROJECT, reference=ROOT / 'data/facebook_main',
    snapshot_holdout=ROOT / 'data/facebook_snapshot_final_holdout',
    prior_independent=ROOT / 'data/generated/facebook_temporal_v3_independent_holdout',
    pilot=ROOT / 'reports/runs/known_k_temporal_gcn_pilot/v1',
    repeats=ROOT / 'reports/runs/known_k_temporal_gcn_repeats/v1',
    s1b=ROOT / 'reports/runs/facebook_main_v2/s1b/seed_7026',
    data=ROOT / 'data/generated/facebook_known_k_temporal_gcn_independent_v1',
    reports=ROOT / 'reports/runs/known_k_temporal_gcn_independent_evaluation/v1',
    generation_config=PROJECT / 'configs/facebook_known_k_temporal_gcn_independent_holdout.yaml',
    notebook=PROJECT / 'notebooks/colab_known_k_temporal_gcn_independent.ipynb',
)
required = [PATHS.reference / 'graph.npz', PATHS.reference / 'train.npz',
            PATHS.reference / 'validation.npz', PATHS.reference / 'test.npz',
            PATHS.snapshot_holdout / 'final_holdout.npz',
            PATHS.prior_independent / 'independent_holdout.npz',
            PATHS.pilot / 'pilot/complete', PATHS.repeats / 'summary/complete',
            PATHS.s1b / 'best_model.pt', PATHS.generation_config, PATHS.notebook,
            PATHS.data / 'graph.npz', PATHS.data / 'independent_holdout.npz',
            PATHS.reports / 'freeze/complete', PATHS.reports / 'seal/complete',
            PATHS.reports / 'opened/complete', PATHS.reports / 'seed_7026/complete',
            AMENDMENT_NOTEBOOK]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(path)
raw = PROJECT / 'data/raw/facebook_combined.txt.gz'
if not raw.is_file():
    import urllib.request
    raw.parent.mkdir(parents=True, exist_ok=True)
    print('Provisioning raw graph only:', 'https://snap.stanford.edu/data/facebook_combined.txt.gz')
    urllib.request.urlretrieve('https://snap.stanford.edu/data/facebook_combined.txt.gz', raw)
print('Original opened reports:', PATHS.reports)
print('Separate amendment root:', PATHS.reports / 'post_open_amendment_v1')
print('Paths checked; no generation or opening performed.')


In [ ]:
CHECK = AMENDMENT._freeze_identity(PATHS, AMENDMENT_NOTEBOOK)
print('Original stage manifests:', CHECK['original_stages'])
print('Dataset unchanged:', CHECK['dataset'])
print('Amendment protocol:', CHECK['protocol'])
print('Read-only check complete. Next cell freezes only the amendment.')


In [ ]:
AMENDMENT_FREEZE = freeze_amendment(PATHS, AMENDMENT_NOTEBOOK)
print('Amendment freeze:', AMENDMENT_FREEZE['status'])
print('Manifest:', PATHS.reports / 'post_open_amendment_v1/freeze/manifest.json')
print('STOP: пришли вывод freeze. 7027 ещё не перезапущен.')


In [ ]:
raise RuntimeError('STOP после amendment freeze: вручную запускай seed_7027 только после проверки')


In [ ]:
AMENDED_7027 = run_amended_seed(PATHS, AMENDMENT_NOTEBOOK, 7027, torch.device(DEVICE))
print('Seed 7027 delta F1:', AMENDED_7027['delta_f1'], 'CI:', AMENDED_7027['f1_ci'])
print('Auxiliary S1b differences:', AMENDED_7027['amendment']['auxiliary_mismatch_counts'])
print('Artifact:', PATHS.reports / 'post_open_amendment_v1/seed_7027')
print('STOP: пришли результат 7027. 7028 не запускай автоматически.')


In [ ]:
raise RuntimeError('STOP после seed_7027: вручную запускай seed_7028 только после проверки')


In [ ]:
AMENDED_7028 = run_amended_seed(PATHS, AMENDMENT_NOTEBOOK, 7028, torch.device(DEVICE))
print('Seed 7028 delta F1:', AMENDED_7028['delta_f1'], 'CI:', AMENDED_7028['f1_ci'])
print('Auxiliary S1b differences:', AMENDED_7028['amendment']['auxiliary_mismatch_counts'])
print('Artifact:', PATHS.reports / 'post_open_amendment_v1/seed_7028')
print('STOP: пришли результат 7028. Summary после проверки.')


In [ ]:
raise RuntimeError('STOP после seed_7028: вручную запускай summary только после проверки')


In [ ]:
AMENDED_SUMMARY = save_amended_summary(PATHS, AMENDMENT_NOTEBOOK)
print('Primary:', AMENDED_SUMMARY['primary'])
print('Mean:', AMENDED_SUMMARY['mean'], 'sample SD:', AMENDED_SUMMARY['sample_sd'])
print('Amendment:', AMENDED_SUMMARY['amendment'])
print('Summary:', PATHS.reports / 'post_open_amendment_v1/summary/payload.json')
print('STOP: пришли summary. Это результат с явно раскрытой поправкой после открытия.')
